# Detection Cache — AMI Group 1

Generates `detections_fusion.json` for all 10 demo sequences using:
- `fusion_rgb.pt`   — RGB component model (YOLOv8n)
- `fusion_event.pt` — Event component model (YOLOv8n)

For each frame both models run independently, then the late-fusion layer
(mirrors `fusion_layer.py`) merges them into three detection types:
- `source='event'`  — event-only (no matching RGB box, conf ≥ 0.5)
- `source='rgb'`    — RGB-only (no matching event box, conf ≥ 0.8)
- `source='fusion'` — event+RGB agreed (IoU > 0.5); confidence = max(event, rgb)

The GUI reads the `source` field to power the Event / RGB / Late Fusion mode buttons.

**Sequences:**
- Test set (KPIs calculated): 8, 9, 12, 20, 21
- Prototyping set (cache only): 84, 85, 124, 127, 201

**Input datasets:**
- `gennepy/ami-fusion-g1` — sequences 8–201 (pre-extracted, double-nested: `N/N/RGB/` etc.)
- `gennepy/ami-fusion-weights` — `fusion_rgb.pt` + `fusion_event.pt` (update this to swap weights)

**Runtime:** GPU T4 × 1 recommended (~15 min total)

**Outputs in `/kaggle/working/`:**
- `sequence_N/detections_fusion.json` × 10 sequences
- `kpis_fusion.json` — mAP50, precision, recall for the 5 test sequences
- `detections_fusion_all.zip` — archive for download


In [ ]:
import bisect, json, re, zipfile
from pathlib import Path
import numpy as np

DATASET  = Path('/kaggle/input/datasets/gennepy/ami-fusion-g1')
WEIGHTS  = Path('/kaggle/input/datasets/gennepy/ami-fusion-weights')
WORK     = Path('/kaggle/working')
TEST_SEQS  = [8, 9, 12, 20, 21]       # KPIs calculated for these
PROTO_SEQS = [84, 85, 124, 127, 201]  # cache only
ALL_SEQS   = TEST_SEQS + PROTO_SEQS
CONF       = 0.1   # inference threshold — matches run_pipeline.py
BATCH      = 32

print("Dataset contents:", sorted(f.name for f in DATASET.iterdir()))
print("Weights:", sorted(f.name for f in WEIGHTS.iterdir()))


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
print('ultralytics ready.')

In [ ]:
from ultralytics import YOLO

rgb_model   = YOLO(str(WEIGHTS / 'fusion_rgb.pt'))
event_model = YOLO(str(WEIGHTS / 'fusion_event.pt'))
print('Models loaded.')
print('  RGB   nc:', rgb_model.model.nc,   'names:', rgb_model.names)
print('  Event nc:', event_model.model.nc, 'names:', event_model.names)


In [ ]:
# ── Late fusion layer (mirrors services/fusion/fusion_layer.py exactly) ────────
EVENT_THRESHOLD = 0.5
RGB_THRESHOLD   = 0.8
IOU_THRESHOLD   = 0.5

def _iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    if inter == 0:
        return 0.0
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)

def fuse_detections(event_dets, rgb_dets):
    final = []
    for ev in event_dets:
        best_iou, best_rgb = 0, None
        for rgb in rgb_dets:
            v = _iou(ev['box'], rgb['box'])
            if v > best_iou:
                best_iou, best_rgb = v, rgb
        if best_rgb and best_iou > IOU_THRESHOLD:
            final.append({'box': ev['box'],
                          'confidence': max(ev['confidence'], best_rgb['confidence']),
                          'source': 'fusion'})
        elif ev['confidence'] >= EVENT_THRESHOLD:
            final.append({'box': ev['box'], 'confidence': ev['confidence'], 'source': 'event'})
    for rgb in rgb_dets:
        if rgb['confidence'] >= RGB_THRESHOLD:
            final.append({'box': rgb['box'], 'confidence': rgb['confidence'], 'source': 'rgb'})
    return final

def _boxes(result):
    return [{'box': b.xyxy[0].tolist(), 'confidence': float(b.conf[0])}
            for b in result.boxes]

def _event_sort_key(p):
    m = re.search(r'_(\d+)\.(png|jpg)$', p.name)
    return int(m.group(1)) if m else 0

print('Fusion layer ready.')

In [ ]:
# ── Ground-truth loader (for KPI evaluation on test sequences) ─────────────────
# coordinates_rgb.txt format: "seconds: x1, y1, x2, y2, class, drone_name"

def load_gt(coord_path):
    gt = []
    for line in Path(coord_path).read_text().splitlines():
        line = line.strip()
        if not line or ':' not in line:
            continue
        ts_str, rest = line.split(':', 1)
        try:
            ts   = float(ts_str.strip())
            vals = [float(x.strip()) for x in rest.split(',')[:4]]
            if len(vals) == 4:
                gt.append((ts, vals))
        except ValueError:
            pass
    return sorted(gt, key=lambda x: x[0])

def _rgb_timestamp(filename):
    # Video_8_17_31_16.463964.jpg  →  seconds since midnight
    stem  = Path(filename).stem
    parts = stem.rsplit('_', 3)
    if len(parts) < 4:
        return None
    try:
        h, m, s = int(parts[1]), int(parts[2]), float(parts[3])
        return h * 3600 + m * 60 + s
    except ValueError:
        return None

def match_gt_to_frames(rgb_files, gt_list, tol=0.05):
    frame_gt = {}
    if not gt_list:
        return frame_gt
    gt_ts = [t for t, _ in gt_list]
    for i, f in enumerate(rgb_files):
        ts = _rgb_timestamp(f.name)
        if ts is None:
            continue
        idx = bisect.bisect_left(gt_ts, ts)
        candidates = []
        if idx > 0:            candidates.append(gt_list[idx - 1])
        if idx < len(gt_list): candidates.append(gt_list[idx])
        if not candidates:
            continue
        closest_ts, closest_box = min(candidates, key=lambda x: abs(x[0] - ts))
        if abs(closest_ts - ts) <= tol:
            frame_gt[i] = closest_box
    return frame_gt

print('GT loader ready.')

In [ ]:
# ── Main detection loop ────────────────────────────────────────────────────────
seq_frame_gt = {}   # {seq_n: {frame_idx: [x1,y1,x2,y2]}} — test seqs only

for seq_n in ALL_SEQS:
    seq_id   = f'sequence_{seq_n}'
    # Kaggle extracts N.zip into N/N/ (double-nested)
    seq_dir  = DATASET / str(seq_n) / str(seq_n)
    out_path = WORK / seq_id / 'detections_fusion.json'
    is_test  = seq_n in TEST_SEQS

    print(f'\n=== {seq_id} ({"test" if is_test else "prototyping"}) ===')

    rgb_dir   = seq_dir / 'RGB'
    event_dir = seq_dir / 'Event' / 'Frames'
    if not event_dir.exists():
        event_dir = seq_dir / 'Event' / 'images'

    rgb_files   = sorted(rgb_dir.glob('*.jpg')) if rgb_dir.exists() else []
    event_files = sorted(
        (event_dir.glob('*.png') if event_dir.exists() else []),
        key=_event_sort_key
    )

    n_frames = min(len(rgb_files), len(event_files))
    print(f'  RGB={len(rgb_files)}  Event={len(event_files)}  paired={n_frames}')
    if n_frames == 0:
        print('  WARNING: no paired frames — skipping')
        continue

    # Load GT for test sequences only
    if is_test:
        coord_path = seq_dir / 'coordinates_rgb.txt'
        gt_list    = load_gt(coord_path) if coord_path.exists() else []
        frame_gt   = match_gt_to_frames(rgb_files[:n_frames], gt_list)
        seq_frame_gt[seq_n] = frame_gt
        print(f'  GT entries={len(gt_list)}  matched frames={len(frame_gt)}')

    # Inference + fusion
    detections = []
    for i in range(0, n_frames, BATCH):
        b_rgb = [str(f) for f in rgb_files[i:i+BATCH]]
        b_evt = [str(f) for f in event_files[i:i+BATCH]]

        rgb_results = rgb_model(b_rgb,   conf=CONF, verbose=False)
        evt_results = event_model(b_evt, conf=CONF, verbose=False)

        for j, (rgb_r, evt_r) in enumerate(zip(rgb_results, evt_results)):
            frame_idx = i + j
            fused = fuse_detections(_boxes(evt_r), _boxes(rgb_r))
            for det in fused:
                x1, y1, x2, y2 = det['box']
                detections.append({
                    'frame':      frame_idx,
                    'bbox':       [x1, y1, x2 - x1, y2 - y1],
                    'confidence': det['confidence'],
                    'class':      'drone',
                    'source':     det['source'],
                })

        if (i // BATCH) % 10 == 0 or i + BATCH >= n_frames:
            print(f'  {min(i+BATCH, n_frames)}/{n_frames} frames  '
                  f'({len(detections)} detections so far)')

    by_src = {}
    for d in detections:
        by_src[d['source']] = by_src.get(d['source'], 0) + 1
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps({
        'sequence_id': seq_id,
        'model':       'fusion',
        'cached':      True,
        'detections':  detections,
    }, indent=2))
    print(f'  → {out_path.relative_to(WORK)}  total={len(detections)}  by_source={by_src}')

print('\n=== All sequences done ===')


In [ ]:
# ── KPI calculation — test sequences only ─────────────────────────────────────
# mAP50: VOC 11-point interpolated AP at IoU threshold 0.5.

def _iou_box(det_box, gt_box):
    # det_box: [x1,y1,w,h]  gt_box: [x1,y1,x2,y2]
    dx1, dy1, dw, dh = det_box
    dx2, dy2 = dx1 + dw, dy1 + dh
    gx1, gy1, gx2, gy2 = gt_box
    ix1, iy1 = max(dx1, gx1), max(dy1, gy1)
    ix2, iy2 = min(dx2, gx2), min(dy2, gy2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    if inter == 0:
        return 0.0
    return inter / (dw * dh + (gx2-gx1)*(gy2-gy1) - inter)

def compute_ap50(detections, frame_gt, iou_thr=0.5):
    n_gt = len(frame_gt)
    if n_gt == 0:
        return {'ap50': None, 'precision': None, 'recall': None, 'n_gt': 0, 'n_det': 0}

    dets = sorted(detections, key=lambda d: d['confidence'], reverse=True)
    matched, tp_list, fp_list = set(), [], []

    for det in dets:
        f = det['frame']
        if f not in frame_gt or f in matched:
            fp_list.append(1); tp_list.append(0)
            continue
        if _iou_box(det['bbox'], frame_gt[f]) >= iou_thr:
            tp_list.append(1); fp_list.append(0)
            matched.add(f)
        else:
            tp_list.append(0); fp_list.append(1)

    tp_cum     = np.cumsum(tp_list)
    fp_cum     = np.cumsum(fp_list)
    recalls    = tp_cum / n_gt
    precisions = tp_cum / (tp_cum + fp_cum)

    ap = sum(
        (precisions[recalls >= thr].max() if (recalls >= thr).any() else 0.0)
        for thr in np.linspace(0, 1, 11)
    ) / 11

    return {
        'ap50':      round(float(ap), 4),
        'precision': round(float(precisions[-1]), 4),
        'recall':    round(float(recalls[-1]),    4),
        'n_gt':      n_gt,
        'n_det':     len(dets),
    }

kpis = {}
all_dets, all_gt, gt_offset = [], {}, 0

for seq_n in TEST_SEQS:
    seq_id   = f'sequence_{seq_n}'
    out_path = WORK / seq_id / 'detections_fusion.json'
    if not out_path.exists():
        print(f'{seq_id}: MISSING — skipping'); continue

    dets     = json.loads(out_path.read_text())['detections']
    frame_gt = seq_frame_gt.get(seq_n, {})
    result   = compute_ap50(dets, frame_gt)
    kpis[seq_id] = result

    for d in dets:
        all_dets.append({**d, 'frame': d['frame'] + gt_offset})
    for f, box in frame_gt.items():
        all_gt[f + gt_offset] = box
    if frame_gt:
        gt_offset += max(frame_gt) + 1

    print(f'{seq_id}: mAP50={result["ap50"]}  P={result["precision"]}  '
          f'R={result["recall"]}  n_gt={result["n_gt"]}  n_det={result["n_det"]}')

overall = compute_ap50(all_dets, all_gt)
kpis['overall'] = overall
print(f'\nOverall (test set): mAP50={overall["ap50"]}  P={overall["precision"]}  '
      f'R={overall["recall"]}  n_gt={overall["n_gt"]}  n_det={overall["n_det"]}')

kpi_path = WORK / 'kpis_fusion.json'
kpi_path.write_text(json.dumps(kpis, indent=2))
print(f'\nKPIs saved → {kpi_path}')

In [ ]:
# ── Package for download ───────────────────────────────────────────────────────
archive = WORK / 'detections_fusion_all.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for seq_n in ALL_SEQS:
        p = WORK / f'sequence_{seq_n}' / 'detections_fusion.json'
        if p.exists():
            zf.write(p, f'sequence_{seq_n}/detections_fusion.json')
    if kpi_path.exists():
        zf.write(kpi_path, 'kpis_fusion.json')

size_mb = archive.stat().st_size / 1024 / 1024
print(f'Archive: {archive.name} ({size_mb:.0f} MB)')
print('Download from: Kaggle → notebook → Output → detections_fusion_all.zip')